***Dependencies***

---



In [ ]:
!pip install -q ultralytics pyyaml

***Code***

---



In [ ]:
import os
import glob
import zipfile
import shutil
import yaml
import urllib.request
from ultralytics import YOLO

# Define setup paths
dataset_dir = "/content/dataset"
target_zip = "/content/TheDataset.zip"
yaml_path = os.path.join(dataset_dir, "data.yaml")
train_txt = os.path.join(dataset_dir, "train.txt")
weights_path = "/content/yolov8n.pt"

# 1. AUTO-DETECT & RENAME ANY .ZIP TO 'TheDataset.zip'
found_zips = glob.glob("/content/*.zip")
other_zips = [f for f in found_zips if os.path.basename(f) != "TheDataset.zip"]

if other_zips:
    if os.path.exists(target_zip):
        os.remove(target_zip)
    os.rename(other_zips[0], target_zip)
    print(f"✅ Renamed '{os.path.basename(other_zips[0])}' ➔ 'TheDataset.zip'")

# 2. AUTO-UNZIP / REORGANIZE INTO /content/dataset
if not os.path.exists(yaml_path):
    print("🔍 Setting up dataset directory...")

    if os.path.exists("/content/data.yaml"):
        os.makedirs(dataset_dir, exist_ok=True)
        for item in ["data.yaml", "train.txt", "images", "labels"]:
            src = os.path.join("/content", item)
            dst = os.path.join(dataset_dir, item)
            if os.path.exists(src) and not os.path.exists(dst):
                shutil.move(src, dst)
        print("✅ Reorganized files into /content/dataset/")

    elif os.path.exists(target_zip):
        print("📦 Extracting 'TheDataset.zip' into /content/dataset/...")
        os.makedirs(dataset_dir, exist_ok=True)
        with zipfile.ZipFile(target_zip, 'r') as zip_ref:
            zip_ref.extractall(dataset_dir)
        print("✅ Dataset successfully extracted!")
    else:
        raise FileNotFoundError("❌ Critical Error: No dataset zip found in /content/")
else:
    print("✅ Dataset directory active and ready!")

# 3. FIX train.txt PATHS
if os.path.exists(train_txt):
    with open(train_txt, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    fixed_lines = []
    for line in lines:
        if not line.startswith("/"):
            clean_path = line.lstrip("./")
            if clean_path.startswith("data/"):
                clean_path = clean_path[5:]
            fixed_path = os.path.join(dataset_dir, clean_path)
        else:
            fixed_path = line
        fixed_lines.append(fixed_path + "\n")

    with open(train_txt, "w") as f:
        f.writelines(fixed_lines)

    print(f"✅ Fixed {len(fixed_lines)} image paths in train.txt")

# 4. CONFIGURE data.yaml
with open(yaml_path, "r") as f:
    config = yaml.safe_load(f)

config["path"] = dataset_dir
config["train"] = "train.txt"
config["val"] = "train.txt"

with open(yaml_path, "w") as f:
    yaml.dump(config, f)

print("✅ data.yaml successfully configured!")

# 5. PRE-DOWNLOAD BASE WEIGHTS
if not os.path.exists(weights_path):
    print("⬇️ Pre-downloading base model weights (yolov8n.pt)...")
    url = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n.pt"
    urllib.request.urlretrieve(url, weights_path)
    print("✅ Base weights successfully downloaded!")
else:
    print("✅ yolov8n.pt already present!")

# 6. START MODEL TRAINING
model = YOLO(weights_path)

results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    project="leaf_disease_run"
)

***Export***

---



In [ ]:
import shutil
from google.colab import files

file = 3

# 1. Zip the train-4 directory
shutil.make_archive(f'train-{file}', 'zip', f'/content/runs/detect/leaf_disease_run/train-{file}')

# 2. Trigger browser download
files.download(f'train-{file}.zip')

***Delete Training Data***

---



In [ ]:
!rm -rf /content/runs/detect/leaf_disease_run/train-2

***Create train.py***

---



In [ ]:
import os
import shutil

# Create src directory
os.makedirs("src", exist_ok=True)

train_code = '''
import os
import yaml
import urllib.request
from ultralytics import YOLO

def train():
    dataset_dir = "/content/dataset"
    yaml_path = os.path.join(dataset_dir, "data.yaml")
    weights_path = "yolov8n.pt"

    # Pre-download base weights if missing
    if not os.path.exists(weights_path):
        print("⬇️ Downloading base model weights (yolov8n.pt)...")
        url = "https://github.com/ultralytics/assets/releases/download/v8.2.0/yolov8n.pt"
        urllib.request.urlretrieve(url, weights_path)

    # Initialize and train
    model = YOLO(weights_path)
    results = model.train(
        data=yaml_path,
        epochs=50,
        imgsz=640,
        batch=16,
        project="leaf_disease_run",
        name="train",
        exist_ok=True
    )
    print("✅ Training complete!")

if __name__ == "__main__":
    train()
'''

with open("src/train.py", "w") as f:
    f.write(train_code)

print("✅ Created src/train.py")

***Create predict.py***

---



In [ ]:
predict_code = '''
import sys
from ultralytics import YOLO

def predict(image_path, weights_path="best.pt"):
    # Load trained model
    model = YOLO(weights_path)

    # Perform inference
    results = model(image_path)

    # Display bounding box output
    results[0].show()

    # Save output image
    results[0].save(filename="prediction_output.jpg")
    print("✅ Prediction saved to prediction_output.jpg")

if __name__ == "__main__":
    img_path = sys.argv[1] if len(sys.argv) > 1 else "test.jpg"
    predict(img_path)
'''

with open("src/predict.py", "w") as f:
    f.write(predict_code)

print("✅ Created src/predict.py")